# WaferAI 재학습 — Colab

로트 단위 train / val / test 3분할로 CoAtNet-0 을 다시 학습해, **오염되지 않은 테스트 정확도**를 얻는다.

* 하이퍼파라미터는 1학기 Colab 학습(`training_reference.py`)과 같다. 바뀐 것은 데이터 분할뿐이다.
* ImageNet 가중치에서 시작한다. 1학기 모델에서 이어 학습하면 그 모델이 이미 본 샘플이 테스트 셋에 섞여 결과가 다시 오염된다.
* 학습 결과는 Drive 에 바로 저장되므로 세션이 끊겨도 가장 좋았던 체크포인트는 남는다(이어서 학습하는 기능은 없다).

**준비**: `WaferAI_colab_bundle.zip` 을 Drive 의 `MyDrive/WaferAI/` 에 올린다.
한 번에 큰 파일을 올리기 어려우면 `build/colab_upload/` 의 조각 파일과 `.sha256` 을 대신 올린다.
런타임 → 런타임 유형 변경 → **GPU(T4)**.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'GPU 가 없습니다. 런타임 유형을 GPU 로 바꾸세요.'
print('PyTorch', torch.__version__, '·', torch.cuda.get_device_name(0))

## 2. Drive 연결과 코드 풀기

In [ ]:
import glob, hashlib, os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/WaferAI'
BUNDLE = f'{DRIVE}/WaferAI_colab_bundle.zip'
parts = sorted(glob.glob(f'{BUNDLE}.part*'))
if not os.path.isfile(BUNDLE) and parts:
    # 나눠 올린 경우: 순서대로 이어 붙인다
    BUNDLE = '/content/WaferAI_colab_bundle.zip'
    with open(BUNDLE, 'wb') as out:
        for part in parts:
            with open(part, 'rb') as handle:
                shutil.copyfileobj(handle, out)
    print(f'조각 {len(parts)}개를 이어 붙였습니다')
assert os.path.isfile(BUNDLE), 'Drive 의 MyDrive/WaferAI/ 에 WaferAI_colab_bundle.zip(또는 조각)이 없습니다'

# 원본 해시가 있으면 대조한다. 조각 하나라도 빠지거나 바뀌면 여기서 멈춘다.
checksum = f'{DRIVE}/WaferAI_colab_bundle.zip.sha256'
if os.path.isfile(checksum):
    expected = open(checksum).read().split()[0]
    actual = hashlib.sha256(open(BUNDLE, 'rb').read()).hexdigest()
    assert actual == expected, f'묶음이 원본과 다릅니다 ({actual[:12]} ≠ {expected[:12]})'
    print('묶음 해시 일치', actual[:12])

!rm -rf /content/waferai && mkdir -p /content/waferai && unzip -q "{BUNDLE}" -d /content/waferai
%cd /content/waferai
!pip -q install timm
!ls

## 3. 데이터 확인

`split_id` 가 **`6685ab33cec9096d`** 여야 한다. 다르면 다른 분할이라 기존 결과와 비교할 수 없다.

In [ ]:
import numpy as np
data = np.load('data/training_set.npz')
split_id = str(data['split_id'])
names = [str(n) for n in data['split_names']]
counts = {name: int((data['split'] == i).sum()) for i, name in enumerate(names)}
print('split_id', split_id, '·', counts)
assert split_id == '6685ab33cec9096d', '분할이 다릅니다. 같은 training_set.npz 를 쓰세요.'

## 4. 학습

결과는 `MyDrive/WaferAI/runs/clean_v1/` 에 저장된다 (`model.pth`, `metrics.json`).
CUDA 에서는 1학기와 같이 혼합 정밀도(AMP)를 쓴다.

In [ ]:
RUN = f'{DRIVE}/runs/clean_v1'
!python -u -m scripts.train --data data/training_set.npz --output {RUN} --workers 2

## 5. 결과

In [ ]:
import json
m = json.load(open(f'{RUN}/metrics.json'))
print(f"최고 검증 정확도 {m['best_val_acc']:.2f}% (epoch {m['best_epoch']})")
for view in ('balanced', 'natural'):
    t = m['test'][view]
    print(f"테스트 {view:9s} n={t['n']:,}  정확도 {t['accuracy']:.4f}  macro-F1 {t['macro_f1']:.4f}")
print()
print('클래스별 F1 (실제 분포):')
for name, f1 in m['test']['natural']['per_class_f1'].items():
    print(f'  {name:10s} {f1:.4f}')

## 6. 다음 단계 (내 컴퓨터에서)

1. Drive 의 `WaferAI/runs/clean_v1/model.pth` 를 내려받아 저장소의 `runs/clean_v1/model.pth` 에 둔다.
2. 승격 판정:
   ```sh
   python3 -m scripts.promote_model --candidate runs/clean_v1/model.pth
   ```
   첫 재학습은 기준 모델(1학기)이 이 테스트 셋을 이미 봤으므로 공정한 비교가 불가능하다.
   판정은 절대 하한(균형 macro-F1 0.85)으로만 하고, 1학기 모델 수치는 참고로만 표시된다.
3. 통과하면 `--apply` 를 붙여 교체한다. 서버에 옮겨 api 를 재시작하면 새 모델이 등록된다.

**Near-full(14개)과 Donut(32개)은 테스트 표본이 작다.** 이 두 클래스의 F1 은 한두 건에 크게 흔들린다.